In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# 1) CONFIG
# ==========================================
CSV_FILES = {
    "Alexis": "Alexis_wingate.csv",
    "Antoine": "Antoine_wingate.csv",
    "Jinwei": "Jinwei_wingate.csv",
    "Victor": "Victor_wingate.csv",
}

XLS_FILE = "Physio_Session2_Wingate.xlsx"
SHEET_NAME = "data"

WINGATE_DURATION = 30      # secondes
BASELINE_BEFORE = 20       # secondes avant effort pour baseline SmO2
SLOPE_WINDOW = 10          # pente sur les 10 premières secondes

# ==========================================
# 2) FONCTIONS
# ==========================================
def load_trainred_csv(file_path):
    """
    Lit un CSV Train.Red et renvoie un DataFrame propre avec colonnes:
    Time, SmO2
    """
    # repérer la vraie ligne d'en-tête
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    header_idx = None
    for i, line in enumerate(lines):
        if "Timestamp (seconds passed)" in line and "SmO2" in line:
            header_idx = i
            break

    if header_idx is None:
        raise ValueError(f"Impossible de trouver l'en-tête SmO2 dans {file_path}")

    df = pd.read_csv(file_path, skiprows=header_idx)
    df.columns = [str(c).strip() for c in df.columns]

    df = df[["Timestamp (seconds passed)", "SmO2"]].copy()
    df.columns = ["Time", "SmO2"]

    df["Time"] = pd.to_numeric(df["Time"], errors="coerce")
    df["SmO2"] = pd.to_numeric(df["SmO2"], errors="coerce")
    df = df.dropna(subset=["Time", "SmO2"]).sort_values("Time").reset_index(drop=True)

    return df


def parse_sync_to_seconds(sync_value, recording_max_time):
    """
    Convertit la valeur Synchro SmO2 en secondes.
    Gère:
    - '00:15:00' -> 900 s
    - '15:00:00' -> si absurde vs durée du csv, on interprète comme 15 min = 900 s
    """
    if pd.isna(sync_value):
        raise ValueError("Synchro SmO2 manquante")

    s = str(sync_value).strip()

    parts = s.split(":")
    if len(parts) != 3:
        raise ValueError(f"Format Synchro SmO2 non reconnu: {s}")

    a, b, c = parts
    a, b, c = int(a), int(b), int(c)

    # interprétation standard hh:mm:ss
    sec_standard = a * 3600 + b * 60 + c

    # si aberrant par rapport à la durée du csv, on tente mm:ss:00
    if sec_standard > recording_max_time + 60:
        # interprétation alternative: premier champ = minutes
        sec_alt = a * 60 + b + c / 60
        return float(sec_alt)

    return float(sec_standard)


def extract_wingate_window(sm_df, start_time_s, duration_s=30, baseline_before_s=20):
    """
    Extrait:
    - baseline SmO2 sur les X secondes avant effort
    - fenêtre Wingate sur 30 s
    Calcule les features utiles
    """
    sm = sm_df.copy()

    # baseline avant effort
    baseline_df = sm[(sm["Time"] >= start_time_s - baseline_before_s) &
                     (sm["Time"] < start_time_s)]
    if baseline_df.empty:
        raise ValueError("Pas de baseline SmO2 avant effort")

    sm0 = float(baseline_df["SmO2"].mean())

    # fenêtre effort
    win = sm[(sm["Time"] >= start_time_s) &
             (sm["Time"] <= start_time_s + duration_s)].copy()

    if len(win) < 5:
        raise ValueError("Fenêtre Wingate trop courte")

    # temps relatif
    win["t_rel"] = win["Time"] - start_time_s

    # désaturation
    win["dSmO2"] = (sm0 - win["SmO2"]).clip(lower=0)

    # dt
    dt = float(np.mean(np.diff(win["Time"])))
    if not np.isfinite(dt) or dt <= 0:
        raise ValueError("dt invalide dans la fenêtre Wingate")

    # AUC sur 30 s
    auc_30 = float(np.sum(win["dSmO2"]) * dt)

    # minimum SmO2
    min_smo2 = float(win["SmO2"].min())

    # chute maximale
    dmax = float(win["dSmO2"].max())

    # pente sur les 10 premières secondes
    early = win[win["t_rel"] <= SLOPE_WINDOW].copy()
    if len(early) >= 3:
        slope_0_10 = float(np.polyfit(early["t_rel"], early["SmO2"], 1)[0])
    else:
        slope_0_10 = np.nan

    return {
        "sm_window": win,
        "SmO2_baseline": sm0,
        "AUC_30s": auc_30,
        "SmO2_min": min_smo2,
        "dSmO2_max": dmax,
        "slope_0_10": slope_0_10
    }


def summarize_lactate_subject(lac_subject):
    """
    À partir des lignes Excel d'un sujet:
    - baseline = Time == -1
    - effort = Time == 0
    - post = Time > 0
    """
    lac_subject = lac_subject.sort_values("Time").copy()

    # baseline
    base_row = lac_subject[lac_subject["Time"] == -1]
    if base_row.empty:
        raise ValueError("Pas de lactate baseline (Time == -1)")
    la_baseline = float(base_row["[La]"].iloc[0])

    # effort immédiat
    effort_row = lac_subject[lac_subject["Time"] == 0]
    la_effort0 = float(effort_row["[La]"].iloc[0]) if not effort_row.empty else np.nan

    # pic post
    post = lac_subject[lac_subject["Time"] > 0]
    if post.empty:
        raise ValueError("Pas de lactates post-effort")
    idx_peak = post["[La]"].idxmax()
    la_peak = float(post.loc[idx_peak, "[La]"])
    t_peak = float(post.loc[idx_peak, "Time"])

    delta_peak = la_peak - la_baseline

    return {
        "La_baseline": la_baseline,
        "La_effort0": la_effort0,
        "La_peak": la_peak,
        "Time_peak_min": t_peak,
        "Delta_La_peak": delta_peak
    }


# ==========================================
# 3) CHARGER L'EXCEL
# ==========================================
lac_all = pd.read_excel(XLS_FILE, sheet_name=SHEET_NAME)
lac_all.columns = [str(c).strip() for c in lac_all.columns]

required_cols = ["Name", "Time", "[La]", "Synchro SmO2"]
for c in required_cols:
    if c not in lac_all.columns:
        raise ValueError(f"Colonne manquante dans l'Excel: {c}")

lac_all["Time"] = pd.to_numeric(lac_all["Time"], errors="coerce")
lac_all["[La]"] = pd.to_numeric(lac_all["[La]"], errors="coerce")

# ==========================================
# 4) EXTRACTION FEATURES + LACTATE
# ==========================================
rows = []
windows = {}

for subject, csv_file in CSV_FILES.items():
    print(f"\n--- {subject} ---")

    # SmO2
    sm = load_trainred_csv(csv_file)
    recording_max = float(sm["Time"].max())

    # lignes Excel du sujet
    lac_sub = lac_all[lac_all["Name"].str.lower() == subject.lower()].copy()
    if lac_sub.empty:
        print(f"{subject}: pas trouvé dans l'Excel")
        continue

    # synchro
    sync_val = lac_sub["Synchro SmO2"].dropna().iloc[0]
    start_time_s = parse_sync_to_seconds(sync_val, recording_max)

    # features SmO2
    sm_feat = extract_wingate_window(
        sm_df=sm,
        start_time_s=start_time_s,
        duration_s=WINGATE_DURATION,
        baseline_before_s=BASELINE_BEFORE
    )

    # résumé lactate
    la_feat = summarize_lactate_subject(lac_sub)

    row = {
        "id": subject,
        "start_time_s": start_time_s,
        **{k: v for k, v in sm_feat.items() if k != "sm_window"},
        **la_feat
    }
    rows.append(row)
    windows[subject] = sm_feat["sm_window"]

feat_df = pd.DataFrame(rows)

print("\n===== FEATURES WINGATE =====")
print(feat_df)

# ==========================================
# 5) MODELE SIMPLE WINGATE
#    Delta_La_peak ~ AUC_30s
# ==========================================
X = np.column_stack([
    np.ones(len(feat_df)),
    feat_df["AUC_30s"].to_numpy(float)
])
y = feat_df["Delta_La_peak"].to_numpy(float)

beta = np.linalg.lstsq(X, y, rcond=None)[0]
feat_df["DeltaLa_pred"] = X @ beta

rmse = float(np.sqrt(np.mean((y - feat_df["DeltaLa_pred"]) ** 2)))
r2 = float(1 - np.sum((y - feat_df["DeltaLa_pred"]) ** 2) / np.sum((y - y.mean()) ** 2))

print("\n===== MODELE WINGATE =====")
print(f"Delta La_peak = {beta[0]:.4f} + {beta[1]:.6f} * AUC_30s")
print(f"RMSE = {rmse:.3f}")
print(f"R²   = {r2:.3f}")

# gain individuel simple
feat_df["gamma_peak_over_auc"] = feat_df["Delta_La_peak"] / feat_df["AUC_30s"]

print("\n===== GAIN INDIVIDUEL =====")
print(feat_df[["id", "AUC_30s", "Delta_La_peak", "gamma_peak_over_auc"]])

# ==========================================
# 6) PLOT MODELE GLOBAL
# ==========================================
plt.figure(figsize=(6, 4))
plt.scatter(feat_df["AUC_30s"], feat_df["Delta_La_peak"], label="Mesuré")
xx = np.linspace(feat_df["AUC_30s"].min(), feat_df["AUC_30s"].max(), 200)
yy = beta[0] + beta[1] * xx
plt.plot(xx, yy, label="Modèle linéaire")
for _, r in feat_df.iterrows():
    plt.annotate(r["id"], (r["AUC_30s"], r["Delta_La_peak"]), textcoords="offset points", xytext=(5, 3))
plt.xlabel("AUC SmO2 sur 30 s")
plt.ylabel("Delta lactate pic (mmol/L)")
plt.title("Wingate — Delta lactate pic ~ AUC SmO2")
plt.grid(True)
plt.legend()
plt.show()

# ==========================================
# 7) PLOT INDIVIDUEL DE CHAQUE WINGATE
# ==========================================
for subject, win in windows.items():
    plt.figure(figsize=(8, 4))
    plt.plot(win["t_rel"], win["SmO2"], label="SmO2")
    plt.xlabel("Temps depuis début Wingate (s)")
    plt.ylabel("SmO2 (%)")
    plt.title(f"{subject} — SmO2 pendant Wingate")
    plt.grid(True)
    plt.legend()
    plt.show()